In [13]:
# MY_CONV_DRIVER.py를 import (현재 directory에 있어야 함)
from MY_CONV_DRIVER import NPUDriver # , LayerConfig
import numpy as np


# NPUDriver instance
driver = NPUDriver("attempt_20.bit")

IMEM Shape: (131072,)
WMEM Shape: (262144,)
OMEM Shape: (131072,)


In [14]:
input_image = np.load('npy_files/input.npy')
weights = {
    'conv1': np.load('npy_files/layer1_0_weight.npy'),
    'conv2': np.load('npy_files/layer2_0_weight.npy'),
    'conv3': np.load('npy_files/layer3_0_weight.npy'),
    'conv4': np.load('npy_files/layer4_0_weight.npy'),
    'fc': np.load('npy_files/fc1_weight.npy')
}

In [15]:
def npu_golden_model_HW(input_image, weights):

    # Conv1 + Leaky ReLU
    x = driver.run_conv_2d(input_image, weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv2 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv3 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv3'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv3 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv4 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv4'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv4 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # MAX Pool
    x = driver.maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    # FC
    output = driver.run_fc_2d(x, weights['fc'], tile_h=3, tile_w=3, tile_oc=1)

    return output.reshape(-1)

In [16]:
import time

start_time = time.time()
#############################################################
print("NPU가 이미지 0 ~ 4번 계산")
for i in range(5):
    output = npu_golden_model_HW(input_image[i], weights)
    print("출력 형태:", output.shape)       # (10,)
    print("출력 값:", output)
#############################################################
end_time = time.time()
runtime = end_time - start_time
print(f"Runtime: {runtime*1000:.3f}ms")

NPU가 이미지 0 ~ 4번 계산
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [127. -94.  16. -30. -73. -26.  -2. -47. -12.   5.]
Conv1

In [12]:
answer = np.load('npy_files/output.npy')
print("정답 형태:", answer[0].shape)
for i in range(5):
    print("정답 레이블:", answer[i])

정답 형태: (10,)
정답 레이블: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
정답 레이블: [  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]
정답 레이블: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]
정답 레이블: [127. -94.  16. -30. -72. -26.  -2. -47. -12.   5.]
정답 레이블: [-47. -50. -17. -69. 127. -13. -78. -12.  -2.  80.]


In [5]:
pj1_input_image = np.load('pj1_npy_files/input.npy')
pj1_input_data = pj1_input_image[0]
pj1_weights = {
    'conv1': np.load('pj1_npy_files/layer1_0_weight.npy'),
    'conv2': np.load('pj1_npy_files/layer2_0_weight.npy'),
    'fc': np.load('pj1_npy_files/fc1_weight.npy')
}

In [6]:
def npu_project1(pj1_input_data, pj1_weights):
    # Conv1 + ReLU
    x = driver.run_conv_2d(pj1_input_data, pj1_weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = driver.relu_loop(x)

    # Conv2 + ReLU
    x = driver.run_conv_2d(x, pj1_weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = driver.relu_loop(x)

    # 최대 풀링
    x = driver.maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    # FC
    output = driver.run_fc_2d(x, pj1_weights['fc'], tile_h=3, tile_w=3, tile_oc=1)
# 
    return output.reshape(-1)

In [7]:
result = npu_project1(input_image[0], pj1_weights)
print(result)
for i in range(11):  # Loop from pj1_input_image[0] to pj1_input_image[10]
    result = npu_project1(pj1_input_image[i], pj1_weights)
    print("출력 값:", result)

Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
[ 21. -35.   8.  27. -10.  11. -38.  82.  -4.  13.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
출력 값: [  4. -18.  23.  60. -39.  81.  19.  31.  21.  22.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
출력 값: [ 99. -70.   8.  12. -16.   0.  24.  19.   2.  41.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
출력 값: [-58. -38.  -3.  14.  37. -26. -41.  16.  18. -29.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
출력 값: [-3. 43. 18. 24.  3. 14. 13. 13. 21. 21.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Maxpool output shape: (16, 12, 12)
출력 값: [-11.   7.   1.  10.  27. -16. -11.  32.  38.  63.]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Max